# minnode — Data Preparation and Algorithm Validation

This notebook prepares the data consumed by the `minnode` dashboard and validates the
shortest-path algorithm used to compute optimal meeting destinations.

## Sections

1. **Data wrangling** — Filter raw GSA City Pair fare data to U.S. CONUS airports,
   join with airport coordinates, and write a single denormalized CSV that the
   dashboard consumes.
2. **Algorithm validation** — Build a graph from the produced CSV, implement
   Dijkstra's shortest-path algorithm, and verify it computes optimal meeting
   points for representative inputs.

## Inputs

- `../data/awards_2026.csv` — GSA City Pair FY 2026 awards (16,073 records).
- `../data/iata-icao.csv` — Airport reference data from `ip2location/ip2location-iata-icao` (CC BY-SA 4.0).

## Output

- `../data/processed/airport_pair_fares.csv` — Denormalized airport-pair fares
  with origin and destination coordinates, filtered to U.S. CONUS.


## Section 1: Data Wrangling

Filters GSA City Pair fare data to U.S. CONUS airports, joins it with airport
coordinates, and writes the denormalized CSV the dashboard consumes. The transform
itself lives in [`scripts/prepare_data.py`](../scripts/prepare_data.py) — the single
source of truth used both here and by the Docker build, so this notebook and the
deployed app can never drift out of sync. See
[Stage 4: Data Wrangling](../docs/04-data-wrangling.md) for the rationale behind
each filtering decision.

In [ ]:
import sys

import pandas as pd

!{sys.executable} ../scripts/prepare_data.py

In [ ]:
pairs = pd.read_csv('../data/processed/airport_pair_fares.csv')
pairs.head()

## Section 2: Algorithm Validation

Build a graph from the produced CSV, implement Dijkstra's shortest-path
algorithm, and verify it computes the optimal meeting destination for a set of
origins.

The algorithm used here is Dijkstra's, not A\*. A\* requires an admissible
heuristic; identifying a suitable heuristic for fare-graph search is deferred to
a later iteration. Implemented as written, the algorithm is mathematically
equivalent to Dijkstra.

### Build the graph

Edges are undirected. The fare column used here is `yca_fare` (the unrestricted
government coach fare); the choice of fare column is a modeling decision
documented in [Stage 6: Statistical Modeling](../docs/06-statistical-modeling.md).

In [6]:
graph = {}
for _, row in pairs.iterrows():
    origin = row['origin']
    destination = row['destination']
    fare = row['yca_fare']

    graph.setdefault(origin, []).append((destination, fare))
    graph.setdefault(destination, []).append((origin, fare))

print(f"Graph nodes: {len(graph):,}")
print(f"Graph edges (directed): {sum(len(v) for v in graph.values()):,}")

Graph nodes: 289
Graph edges (directed): 21,870


### Dijkstra's algorithm

Compute the shortest path from a starting airport to every other reachable airport
in the graph. Returns the cost map (airport → minimum cost from start).

In [7]:
import heapq
import math

def dijkstra(graph, start):
    """Compute shortest-path costs from `start` to every reachable node."""
    costs = {node: math.inf for node in graph}
    costs[start] = 0
    queue = [(0, start)]

    while queue:
        current_cost, current_node = heapq.heappop(queue)
        if current_cost > costs[current_node]:
            continue
        for neighbor, weight in graph[current_node]:
            new_cost = current_cost + weight
            if new_cost < costs[neighbor]:
                costs[neighbor] = new_cost
                heapq.heappush(queue, (new_cost, neighbor))

    return costs

### Compute optimal meeting point

For a set of origin airports, find the destination that minimizes the sum of
shortest-path costs from each origin. This is the artifact's core computation.

In [8]:
def optimal_meeting_point(graph, origins):
    """Return (best_destination, total_cost, per_origin_costs)."""
    cost_maps = {origin: dijkstra(graph, origin) for origin in origins}

    candidates = set(graph.keys())
    best_destination = None
    best_total = math.inf

    for candidate in candidates:
        total = sum(cost_maps[origin].get(candidate, math.inf) for origin in origins)
        if total < best_total:
            best_total = total
            best_destination = candidate

    per_origin = {origin: cost_maps[origin][best_destination] for origin in origins}
    return best_destination, best_total, per_origin

### Validation: representative example

In [9]:
origins = ['MSP', 'DCA', 'LAX']
destination, total_cost, per_origin = optimal_meeting_point(graph, origins)

print(f"Origins: {origins}")
print(f"Optimal meeting destination: {destination}")
print(f"Total cost: ${total_cost:,.2f}")
print()
print("Per-origin costs:")
for origin, cost in per_origin.items():
    print(f"  {origin} -> {destination}: ${cost:,.2f}")

Origins: ['MSP', 'DCA', 'LAX']
Optimal meeting destination: ORD
Total cost: $352.00

Per-origin costs:
  MSP -> ORD: $89.00
  DCA -> ORD: $114.00
  LAX -> ORD: $149.00


### Ranked candidates

Compute the total cost for every candidate destination and display the top
candidates ranked by total cost. Held in memory; not written to disk.

In [10]:
cost_maps = {origin: dijkstra(graph, origin) for origin in origins}

candidates = []
for candidate in graph.keys():
    total = sum(cost_maps[origin].get(candidate, math.inf) for origin in origins)
    if math.isfinite(total):
        candidates.append((candidate, total))

ranked = pd.DataFrame(
    sorted(candidates, key=lambda x: x[1]),
    columns=['destination', 'total_cost'],
)
ranked.head(10)

,destination,total_cost
0,ORD,352
1,MSP,425
2,DEN,448
3,DCA,466
4,EWR,481
5,LAX,485
6,SAT,517
7,BOS,522
8,GRR,529
9,MCI,539
